# H5: Metacognitive Monitoring of the Foraging Computation

Tests whether anxiety and confidence — as metacognitive monitors — track the first-order survival computation and predict foraging efficiency beyond ω and κ.

- **H5a:** Anxiety calibration → optimality (beyond ω + κ)
- **H5b:** Anxiety slope → adaptive choice shifting
- **H5c:** ω → confidence, not anxiety
- **H5d:** Confidence → error type (overcautious vs reckless)

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import pearsonr, zscore, linregress
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

EXCLUDE = [154, 197, 208]
DATA_DIR = Path('data/exploratory_350/processed/stage5_filtered_data_20260320_191950')
print(f'Repo root: {REPO_ROOT}')

Repo root: /workspace


## Load CM2 parameters (ω, κ)

In [2]:
# Load pre-fitted CM2 parameters from master_subject_df
master_path = Path('results/stats/joint_optimal/master_subject_df.csv')
cm2_params = pd.read_csv(master_path, index_col='subj')[['omega', 'kappa']]
subjects = cm2_params.index.tolist()
omega = cm2_params['omega'].values
kappa = cm2_params['kappa'].values

print(f'Loaded CM2 params: {len(subjects)} subjects')
print(f'  ω: mean={omega.mean():.3f}, SD={omega.std():.3f}')
print(f'  κ: mean={kappa.mean():.3f}, SD={kappa.std():.3f}')

Loaded CM2 params: 290 subjects
  ω: mean=1.722, SD=2.475
  κ: mean=0.482, SD=0.771


## Build master subject dataframe

In [3]:
# Load behavioral + affect + clinical data
beh = pd.read_csv(DATA_DIR / 'behavior_rich.csv', low_memory=False)
beh = beh[~beh['subj'].isin(EXCLUDE)]
cdf = beh[beh['type'] == 1].copy()
cdf['T_round'] = cdf['threat'].round(1)
feelings = pd.read_csv(DATA_DIR / 'feelings.csv')
feelings = feelings[~feelings['subj'].isin(EXCLUDE)]
psych = pd.read_csv(DATA_DIR / 'psych.csv')
psych = psych[~psych['subj'].isin(EXCLUDE)].set_index('subj')

# Affect decomposition
anx = feelings[feelings['questionLabel'] == 'anxiety'][['subj', 'threat', 'response']].dropna()
conf = feelings[feelings['questionLabel'] == 'confidence'][['subj', 'threat', 'response']].dropna()

affect = {}
for s in subjects:
    sa = anx[anx['subj'] == s]; sc = conf[conf['subj'] == s]
    m = {}
    if len(sa) >= 5:
        sl, ic, r_cal, _, _ = linregress(sa['threat'], sa['response'])
        m['mean_anx'] = sa['response'].mean()
        m['anx_slope'] = sl
        m['calibration'] = r_cal
        m['anx_T01'] = sa[sa['threat'].round(1) == 0.1]['response'].mean()
        m['anx_T09'] = sa[sa['threat'].round(1) == 0.9]['response'].mean()
    if len(sc) >= 5:
        m['mean_conf'] = sc['response'].mean()
        m['conf_slope'] = linregress(sc['threat'], sc['response'])[0]
    affect[s] = m
affect_df = pd.DataFrame(affect).T; affect_df.index.name = 'subj'

# Behavioral outcomes
attack = beh[beh['isAttackTrial'] == 1]
escape_rate = attack.groupby('subj').apply(lambda x: (x['trialEndState'] == 'escaped').mean()).rename('escape_rate')
earnings = beh.groupby('subj')['trialReward'].sum().rename('earnings')
p_heavy = cdf.groupby('subj')['choice'].mean().rename('p_heavy')

# Choice/vigor shifts
p_heavy_T01 = cdf[cdf['T_round'] == 0.1].groupby('subj')['choice'].mean()
p_heavy_T09 = cdf[cdf['T_round'] == 0.9].groupby('subj')['choice'].mean()
choice_shift = (p_heavy_T01 - p_heavy_T09).rename('choice_shift')

cells = pd.read_csv('results/stats/vigor_analysis/cell_means.csv')
vigor_T01 = cells[cells['T_round'] == 0.1].groupby('subj')['mean_rate'].mean()
vigor_T09 = cells[cells['T_round'] == 0.9].groupby('subj')['mean_rate'].mean()
vigor_shift = (vigor_T09 - vigor_T01).rename('vigor_shift')

# Optimality
opt_map = {}
for (T, D), grp in cdf.groupby(['T_round', 'distance_H']):
    er_h = grp[grp['choice'] == 1]['trialReward'].mean() if (grp['choice'] == 1).sum() > 0 else -99
    er_l = grp[grp['choice'] == 0]['trialReward'].mean() if (grp['choice'] == 0).sum() > 0 else -99
    opt_map[(T, D)] = 1 if er_h > er_l else 0
cdf['optimal'] = cdf.apply(lambda r: opt_map.get((r['T_round'], r['distance_H']), np.nan), axis=1)
cdf['is_opt'] = (cdf['choice'] == cdf['optimal']).astype(int)
cdf['err_type'] = np.where(cdf['is_opt'] == 1, 'correct',
                           np.where(cdf['choice'] == 0, 'overcautious', 'reckless'))
subj_opt = cdf.groupby('subj').agg(
    pct_opt=('is_opt', 'mean'),
    n_oc=('err_type', lambda x: (x == 'overcautious').sum()),
    n_rk=('err_type', lambda x: (x == 'reckless').sum())
)

# Build master
master = pd.DataFrame({'subj': subjects, 'omega': omega, 'kappa': kappa}).set_index('subj')
master['log_omega'] = np.log(master['omega'])
master['log_kappa'] = np.log(master['kappa'])
for s in [escape_rate, earnings, p_heavy, choice_shift, vigor_shift]:
    master = master.join(s, how='left')
master = master.join(affect_df, how='left')
master = master.join(subj_opt, how='left')
master['p_heavy_T01'] = p_heavy_T01
master['p_heavy_T09'] = p_heavy_T09

# Z-scores
for c in ['log_omega', 'log_kappa', 'calibration', 'anx_slope', 'mean_anx', 'mean_conf']:
    if c in master.columns:
        v = master[c].dropna()
        master.loc[v.index, c + '_z'] = zscore(v.values)
master['omega_z'] = master['log_omega_z']
master['kappa_z'] = master['log_kappa_z']

print(f'Master: {len(master)} subjects')
print(f'  Escape rate: {master["escape_rate"].mean():.3f}')
print(f'  Earnings: {master["earnings"].mean():.1f}')
print(f'  Calibration: {master["calibration"].mean():.3f}')

Master: 290 subjects
  Escape rate: 0.384
  Earnings: 6.5
  Calibration: 0.315


## H5a: Anxiety calibration → optimality beyond ω + κ

In [4]:
v = master.dropna(subset=['omega_z', 'kappa_z', 'calibration_z'])
print('H5a — Calibration predicts optimality beyond ω + κ')
print(f'N = {len(v)}\n')

results_h5a = {}
for outcome, label in [('pct_opt', '% Optimal'), ('escape_rate', 'Escape'), ('earnings', 'Earnings')]:
    if outcome not in v.columns: continue
    y = v[outcome].values
    vm = np.isfinite(y)
    X1 = sm.add_constant(v[['omega_z', 'kappa_z']].values[vm])
    m1 = sm.OLS(y[vm], X1).fit()
    X2 = sm.add_constant(v[['omega_z', 'kappa_z', 'calibration_z']].values[vm])
    m2 = sm.OLS(y[vm], X2).fit()
    delta = m2.rsquared - m1.rsquared
    passed = delta > 0.03 and m2.pvalues[3] < 0.01
    results_h5a[outcome] = {'r2_base': m1.rsquared, 'r2_cal': m2.rsquared,
                            'delta': delta, 'p': m2.pvalues[3], 'passed': passed}
    status = '✓' if passed else '✗'
    print(f'  {label}: R²(ω+κ)={m1.rsquared:.4f} → R²(+cal)={m2.rsquared:.4f} '
          f'(ΔR²={delta:+.4f}, cal p={m2.pvalues[3]:.4f}) {status}')

n_pass = sum(1 for v in results_h5a.values() if v['passed'])
print(f'\nH5a PASS: {"✓" if n_pass >= 2 else "✗"} ({n_pass}/3 outcomes, threshold: ≥2)')

H5a — Calibration predicts optimality beyond ω + κ
N = 286

  % Optimal: R²(ω+κ)=0.4727 → R²(+cal)=0.5440 (ΔR²=+0.0713, cal p=0.0000) ✓
  Escape: R²(ω+κ)=0.0591 → R²(+cal)=0.0956 (ΔR²=+0.0364, cal p=0.0009) ✓
  Earnings: R²(ω+κ)=0.0115 → R²(+cal)=0.0682 (ΔR²=+0.0567, cal p=0.0000) ✓

H5a PASS: ✓ (3/3 outcomes, threshold: ≥2)


## H5b: Anxiety slope → adaptive choice shifting

In [5]:
v = master.dropna(subset=['anx_slope', 'choice_shift', 'vigor_shift'])
print('H5b — Anxiety reactivity predicts choice adaptation')
print(f'N = {len(v)}\n')

r_cs, p_cs = pearsonr(v['anx_slope'], v['choice_shift'])
r_vs, p_vs = pearsonr(v['anx_slope'], v['vigor_shift'])

passed = r_cs > 0.20 and p_cs < 0.01
print(f'  r(anxiety slope, choice shift) = {r_cs:+.3f} (p={p_cs:.4f})')
print(f'  r(anxiety slope, vigor shift)  = {r_vs:+.3f} (p={p_vs:.4f})')
print(f'\nH5b PASS: {"✓" if passed else "✗"} (threshold: r > .20, p < .01)')

H5b — Anxiety reactivity predicts choice adaptation
N = 290

  r(anxiety slope, choice shift) = +0.389 (p=0.0000)
  r(anxiety slope, vigor shift)  = +0.004 (p=0.9490)

H5b PASS: ✓ (threshold: r > .20, p < .01)


## H5c: ω → confidence, not anxiety

In [6]:
v = master.dropna(subset=['omega_z', 'mean_anx', 'mean_conf'])
print('H5c — ω maps to confidence, not anxiety')
print(f'N = {len(v)}\n')

r_oc, p_oc = pearsonr(v['omega_z'], v['mean_conf'])
r_oa, p_oa = pearsonr(v['omega_z'], v['mean_anx'])
r_kc, p_kc = pearsonr(v['kappa_z'], v['mean_conf'])
r_ka, p_ka = pearsonr(v['kappa_z'], v['mean_anx'])

pass_conf = r_oc < 0 and p_oc < 0.01
pass_anx = abs(r_oa) < 0.10
passed = pass_conf and pass_anx

print(f'  r(ω, confidence) = {r_oc:+.3f} (p={p_oc:.4f}) {"✓" if pass_conf else "✗"}')
print(f'  r(ω, anxiety)    = {r_oa:+.3f} (p={p_oa:.4f}) {"✓" if pass_anx else "✗"}')
print(f'  r(κ, confidence) = {r_kc:+.3f} (p={p_kc:.4f})')
print(f'  r(κ, anxiety)    = {r_ka:+.3f} (p={p_ka:.4f})')
print(f'\nH5c PASS: {"✓" if passed else "✗"}')

H5c — ω maps to confidence, not anxiety
N = 290

  r(ω, confidence) = -0.203 (p=0.0005) ✓
  r(ω, anxiety)    = +0.076 (p=0.1978) ✓
  r(κ, confidence) = -0.161 (p=0.0061)
  r(κ, anxiety)    = +0.035 (p=0.5489)

H5c PASS: ✓


## H5d: Confidence → error type

In [7]:
v = master.dropna(subset=['mean_conf', 'n_oc', 'n_rk'])
print('H5d — Confidence predicts error type, not error rate')
print(f'N = {len(v)}\n')

r_oc_err, p_oc_err = pearsonr(v['mean_conf'], v['n_oc'])
r_rk_err, p_rk_err = pearsonr(v['mean_conf'], v['n_rk'])

pass_oc = r_oc_err < 0 and p_oc_err < 0.01
pass_rk = r_rk_err > 0 and p_rk_err < 0.01
passed = pass_oc and pass_rk

print(f'  r(confidence, n_overcautious) = {r_oc_err:+.3f} (p={p_oc_err:.4f}) {"✓" if pass_oc else "✗"}')
print(f'  r(confidence, n_reckless)     = {r_rk_err:+.3f} (p={p_rk_err:.4f}) {"✓" if pass_rk else "✗"}')
print(f'\nH5d PASS: {"✓" if passed else "✗"}')

H5d — Confidence predicts error type, not error rate
N = 290

  r(confidence, n_overcautious) = -0.224 (p=0.0001) ✓
  r(confidence, n_reckless)     = +0.200 (p=0.0006) ✓

H5d PASS: ✓


In [8]:
print('=' * 70)
print('H5 SUMMARY')
print('=' * 70)

tests = [
    ('H5a', 'Cal ΔR² optimality', '> .03, p < .01',
     f'+{results_h5a["pct_opt"]["delta"]:.3f} (p={results_h5a["pct_opt"]["p"]:.4f})',
     results_h5a['pct_opt']['passed']),
    ('H5a', 'Cal ΔR² earnings', '> .03, p < .01',
     f'+{results_h5a["earnings"]["delta"]:.3f} (p={results_h5a["earnings"]["p"]:.4f})',
     results_h5a['earnings']['passed']),
    ('H5b', 'Slope → choice shift', 'r > .20, p < .01',
     f'r={r_cs:+.3f} (p={p_cs:.4f})', r_cs > 0.20 and p_cs < 0.01),
    ('H5c', 'ω → confidence', 'r < 0, p < .01',
     f'r={r_oc:+.3f} (p={p_oc:.4f})', r_oc < 0 and p_oc < 0.01),
    ('H5c', 'ω → anxiety', '|r| < .10',
     f'r={r_oa:+.3f}', abs(r_oa) < 0.10),
    ('H5d', 'Conf → overcautious', 'r < 0, p < .01',
     f'r={r_oc_err:+.3f} (p={p_oc_err:.4f})', r_oc_err < 0 and p_oc_err < 0.01),
    ('H5d', 'Conf → reckless', 'r > 0, p < .01',
     f'r={r_rk_err:+.3f} (p={p_rk_err:.4f})', r_rk_err > 0 and p_rk_err < 0.01),
]

print(f'\n{"Hyp":<6} {"Test":<25} {"Threshold":<20} {"Result":<30} {"Pass":<5}')
print('-' * 86)
for hyp, test, thresh, result, passed in tests:
    print(f'{hyp:<6} {test:<25} {thresh:<20} {result:<30} {"✓" if passed else "✗"}')

n_total = len(tests)
n_pass = sum(1 for _, _, _, _, p in tests if p)
print(f'\n{n_pass}/{n_total} tests pass.')

H5 SUMMARY

Hyp    Test                      Threshold            Result                         Pass 
--------------------------------------------------------------------------------------
H5a    Cal ΔR² optimality        > .03, p < .01       +0.071 (p=0.0000)              ✓
H5a    Cal ΔR² earnings          > .03, p < .01       +0.057 (p=0.0000)              ✓
H5b    Slope → choice shift      r > .20, p < .01     r=+0.389 (p=0.0000)            ✓
H5c    ω → confidence            r < 0, p < .01       r=-0.203 (p=0.0005)            ✓
H5c    ω → anxiety               |r| < .10            r=+0.076                       ✓
H5d    Conf → overcautious       r < 0, p < .01       r=-0.224 (p=0.0001)            ✓
H5d    Conf → reckless           r > 0, p < .01       r=+0.200 (p=0.0006)            ✓

7/7 tests pass.
